# PEFT v4 validation-only inference diagnostics

This notebook diagnoses two remaining inference differences in the completed answer-only PEFT v4 experiment without retraining:

1. **Condition B — BOS-corrected:** the existing 20-label inference prompt, tokenized with `add_special_tokens=False` so the chat template supplies exactly one beginning-of-sequence token.
2. **Condition C — training-matched:** the exact prompt construction used during v4 training, also tokenized with `add_special_tokens=False`.

The already-saved v4 validation predictions are Condition A. Every new condition writes to a separate resumable file. The notebook loads only validation data, never loads the test split, never trains, and never overwrites the completed v4 predictions or adapter.


## 1. Environment and frozen inputs


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

%pip install -q transformers accelerate bitsandbytes peft scikit-learn


In [ ]:
import gc
import json
import os
import random
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import transformers
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DATA_PATH = "/content/drive/MyDrive/Thesis1_Project"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_DIR = os.path.join(DATA_PATH, "peft_answer_only_control_v4_final_adapter")
SEED = 42

# Condition A is immutable: this is the validation output already produced by v4.
CONDITION_A_FINAL = os.path.join(
    DATA_PATH, "peft_answer_only_control_v4_validation_final.csv"
)

CONDITIONS = {
    "B_label_prompt_single_bos": {
        "partial": os.path.join(DATA_PATH, "peft_v4_validation_label_prompt_single_bos_partial.csv"),
        "final": os.path.join(DATA_PATH, "peft_v4_validation_label_prompt_single_bos_final.csv"),
    },
    "C_training_prompt_single_bos": {
        "partial": os.path.join(DATA_PATH, "peft_v4_validation_training_prompt_single_bos_partial.csv"),
        "final": os.path.join(DATA_PATH, "peft_v4_validation_training_prompt_single_bos_final.csv"),
    },
}

SUMMARY_PATH = os.path.join(DATA_PATH, "peft_v4_validation_diagnostic_comparison.csv")
PER_CLASS_PATH = os.path.join(DATA_PATH, "peft_v4_validation_per_class_comparison.csv")
TRANSITIONS_PATH = os.path.join(DATA_PATH, "peft_v4_validation_pair_transitions.csv")
CONFLICT_PATH = os.path.join(DATA_PATH, "peft_v4_validation_conflict_analysis.csv")
METADATA_PATH = os.path.join(DATA_PATH, "peft_v4_validation_diagnostic_metadata.json")

CHOICES = [
    "Parties", "License Grant", "Cap On Liability", "Audit Rights",
    "Anti-Assignment", "Insurance", "Document Name", "Agreement Date",
    "Governing Law", "Expiration Date", "Effective Date",
    "Post-Termination Services", "Revenue/Profit Sharing",
    "Minimum Commitment", "Exclusivity", "Rofr/Rofo/Rofn",
    "Ip Ownership Assignment", "Non-Transferable License", "Non-Compete",
    "Termination For Convenience",
]
CHOICES_STR = ", ".join(CHOICES)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
torch.cuda.manual_seed_all(SEED)

required_files = [
    os.path.join(DATA_PATH, "val.csv"),
    os.path.join(DATA_PATH, "peft_val.jsonl"),
    CONDITION_A_FINAL,
    os.path.join(ADAPTER_DIR, "adapter_config.json"),
]
missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    raise FileNotFoundError("Required frozen files are missing:\n" + "\n".join(missing))

val_df = pd.read_csv(os.path.join(DATA_PATH, "val.csv")).reset_index(drop=True)
raw_val = pd.read_json(os.path.join(DATA_PATH, "peft_val.jsonl"), lines=True).reset_index(drop=True)
condition_a = pd.read_csv(CONDITION_A_FINAL)

if len(val_df) != 833 or len(raw_val) != 833 or len(condition_a) != 833:
    raise ValueError("Expected exactly 833 rows in every validation artifact.")
if set(val_df["clause_type"].unique()) != set(CHOICES):
    raise ValueError("Validation labels do not match the frozen 20-class schema.")
if not raw_val["input"].astype(str).equals(val_df["clause_text"].astype(str)):
    raise ValueError("peft_val.jsonl inputs are not aligned with val.csv.")
if not raw_val["output"].astype(str).equals(val_df["clause_type"].astype(str)):
    raise ValueError("peft_val.jsonl labels are not aligned with val.csv.")

condition_a["row_id"] = pd.to_numeric(condition_a["row_id"], errors="raise").astype(int)
condition_a = condition_a.drop_duplicates("row_id", keep="last").sort_values("row_id")
if set(condition_a["row_id"]) != set(range(833)):
    raise ValueError("Condition A row IDs are incomplete or misaligned.")
if not condition_a["clause_text"].astype(str).reset_index(drop=True).equals(
    val_df["clause_text"].astype(str)
):
    raise ValueError("Condition A clause text is not aligned with val.csv.")
if not condition_a["clause_type"].astype(str).reset_index(drop=True).equals(
    val_df["clause_type"].astype(str)
):
    raise ValueError("Condition A labels are not aligned with val.csv.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Transformers: {transformers.__version__}")
print("Validation rows and all three frozen artifacts are aligned: 833")
print("No test split has been loaded.")


## 2. Reconstruct the two diagnostic prompts


In [ ]:
def build_chat_prompt(instruction):
    messages = [{"role": "user", "content": instruction}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"<s>[INST] {instruction} [/INST]"


def build_label_list_prompt(clause_text):
    return build_chat_prompt(
        "Classify the legal clause below into exactly ONE of these categories:\n"
        f"{CHOICES_STR}\n\n"
        "Return ONLY the category name. Do not explain your answer and do not add punctuation.\n\n"
        f"Clause:\n{clause_text}"
    )


def build_training_matched_prompt(raw_example):
    instruction = (
        f"{raw_example['instruction']}\n\n"
        f"Clause:\n{raw_example['input']}\n\n"
        "Return ONLY the category name. Do not explain your answer and do not add punctuation."
    )
    return build_chat_prompt(instruction)


sample_b = build_label_list_prompt(val_df.loc[0, "clause_text"])
sample_c = build_training_matched_prompt(raw_val.loc[0])

def token_audit(name, prompt, add_special_tokens):
    ids = tokenizer(prompt, add_special_tokens=add_special_tokens)["input_ids"]
    bos_count = ids.count(tokenizer.bos_token_id)
    print(
        f"{name}: tokens={len(ids)}, BOS count={bos_count}, "
        f"first tokens={tokenizer.convert_ids_to_tokens(ids[:8])}"
    )
    return ids, bos_count


_, a_bos = token_audit("A reconstruction: label prompt + tokenizer default", sample_b, True)
_, b_bos = token_audit("B: label prompt + add_special_tokens=False", sample_b, False)
_, c_bos = token_audit("C: training prompt + add_special_tokens=False", sample_c, False)

if b_bos != 1 or c_bos != 1:
    raise RuntimeError(
        "The corrected prompt does not contain exactly one BOS token. Stop and inspect the token audit."
    )

print("\nCondition B prompt preview:\n", sample_b[:700])
print("\nCondition C prompt preview:\n", sample_c[:700])
print(f"\nDefault reconstruction BOS count: {a_bos}")
print("Corrected conditions B and C each contain exactly one BOS token.")


## 3. Generate validation predictions for Conditions B and C

Both conditions use the same completed adapter, deterministic decoding, strict parser and validation rows. Each condition checkpoints after every batch and resumes from its own partial file. The existing Condition A file is read-only.


In [ ]:
def parse_prediction(raw_output):
    if raw_output is None or pd.isna(raw_output):
        return "Unknown"
    text = str(raw_output).strip()
    if not text:
        return "Unknown"
    candidates = [text, text.splitlines()[0].strip()]
    for candidate in candidates:
        candidate = re.sub(r"^[\s`*_\"'“”‘’]+|[\s`*_\"'“”‘’]+$", "", candidate).strip()
        category_match = re.fullmatch(r"(?i)category\s*:\s*(.+)", candidate)
        if category_match:
            candidate = category_match.group(1).strip()
        candidate = re.sub(r"[.,;:!?]+$", "", candidate).strip()
        for choice in CHOICES:
            if candidate.casefold() == choice.casefold():
                return choice
    return "Unknown"


def validate_predictions(frame, condition_name):
    frame = frame.copy()
    frame["row_id"] = pd.to_numeric(frame["row_id"], errors="raise").astype(int)
    frame = frame.drop_duplicates("row_id", keep="last").sort_values("row_id")
    if len(frame) != len(val_df) or set(frame["row_id"]) != set(range(len(val_df))):
        raise RuntimeError(f"{condition_name}: predictions are incomplete or row IDs are misaligned.")
    if not frame["clause_text"].astype(str).reset_index(drop=True).equals(
        val_df["clause_text"].astype(str)
    ):
        raise RuntimeError(f"{condition_name}: clause text is misaligned.")
    if not frame["clause_type"].astype(str).reset_index(drop=True).equals(
        val_df["clause_type"].astype(str)
    ):
        raise RuntimeError(f"{condition_name}: reference labels are misaligned.")
    return frame


def run_condition(condition_name, prompt_builder, evaluation_model, batch_size=4):
    paths = CONDITIONS[condition_name]
    if os.path.exists(paths["final"]):
        result = pd.read_csv(paths["final"])
        result = validate_predictions(result, condition_name)
        print(f"{condition_name}: complete final predictions already exist.")
        return result

    if os.path.exists(paths["partial"]):
        result = pd.read_csv(paths["partial"])
        result["row_id"] = pd.to_numeric(result["row_id"], errors="coerce").astype("Int64")
        result = result.dropna(subset=["row_id"]).drop_duplicates("row_id", keep="last")
        print(f"{condition_name}: resuming at {len(result):,}/{len(val_df):,} rows.")
    else:
        result = pd.DataFrame(
            columns=["row_id", "clause_text", "clause_type", "predicted_label", "raw_output"]
        )

    completed_ids = set(result["row_id"].astype(int)) if len(result) else set()
    pending = val_df.loc[~val_df.index.isin(completed_ids)].copy()

    for start in range(0, len(pending), batch_size):
        batch = pending.iloc[start:start + batch_size]
        prompts = [prompt_builder(int(row_id)) for row_id in batch.index]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=4096,
            add_special_tokens=False,
        ).to("cuda")

        # Every non-padding prompt must contain exactly one BOS token.
        for row_ids, attention in zip(inputs["input_ids"], inputs["attention_mask"]):
            active_ids = row_ids[attention.bool()].tolist()
            if active_ids.count(tokenizer.bos_token_id) != 1:
                raise RuntimeError(f"{condition_name}: a batched prompt does not contain exactly one BOS token.")

        with torch.inference_mode():
            generated = evaluation_model.generate(
                **inputs,
                max_new_tokens=12,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_tokens = generated[:, inputs["input_ids"].shape[1]:]
        raw_outputs = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        rows = []
        for row_id, (_, row), raw_output in zip(batch.index, batch.iterrows(), raw_outputs):
            rows.append({
                "row_id": int(row_id),
                "clause_text": row["clause_text"],
                "clause_type": row["clause_type"],
                "predicted_label": parse_prediction(raw_output),
                "raw_output": raw_output.strip(),
            })

        result = pd.concat([result, pd.DataFrame(rows)], ignore_index=True)
        result["row_id"] = result["row_id"].astype(int)
        result = result.drop_duplicates("row_id", keep="last").sort_values("row_id")
        result.to_csv(paths["partial"], index=False)
        if len(result) % 100 < batch_size or len(result) == len(val_df):
            print(f"{condition_name}: {len(result):,}/{len(val_df):,}")

    result = validate_predictions(result, condition_name)
    result.to_csv(paths["final"], index=False)
    print(f"{condition_name}: saved {paths['final']}")
    return result


In [ ]:
for variable_name in ["model", "base_model", "evaluation_model"]:
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    dtype=torch.float16,
)
evaluation_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)
evaluation_model.eval()
evaluation_model.config.use_cache = True

condition_b = run_condition(
    "B_label_prompt_single_bos",
    lambda row_id: build_label_list_prompt(val_df.loc[row_id, "clause_text"]),
    evaluation_model,
)
condition_c = run_condition(
    "C_training_prompt_single_bos",
    lambda row_id: build_training_matched_prompt(raw_val.loc[row_id]),
    evaluation_model,
)

del evaluation_model, base_model
gc.collect()
torch.cuda.empty_cache()
print("Validation inference complete. No test data were loaded or evaluated.")


## 4. Compare Conditions A, B and C


In [ ]:
condition_a = validate_predictions(condition_a, "A_existing_label_prompt_default_bos")
condition_b = validate_predictions(condition_b, "B_label_prompt_single_bos")
condition_c = validate_predictions(condition_c, "C_training_prompt_single_bos")

results = {
    "A_existing_label_prompt_default_bos": condition_a,
    "B_label_prompt_single_bos": condition_b,
    "C_training_prompt_single_bos": condition_c,
}

def metric_row(name, frame):
    y_true = frame["clause_type"]
    y_pred = frame["predicted_label"]
    return {
        "condition": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true, y_pred, labels=CHOICES, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred, labels=CHOICES, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, labels=CHOICES, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, labels=CHOICES, average="weighted", zero_division=0
        ),
        "unknown_count": int(y_pred.eq("Unknown").sum()),
        "unknown_rate": float(y_pred.eq("Unknown").mean()),
        "rows": int(len(frame)),
        "test_evaluated": False,
    }


summary = pd.DataFrame([metric_row(name, frame) for name, frame in results.items()])
reference_macro_f1 = summary.loc[
    summary["condition"].eq("A_existing_label_prompt_default_bos"), "macro_f1"
].iloc[0]
summary["delta_macro_f1_vs_A"] = summary["macro_f1"] - reference_macro_f1
summary["material_abs_delta_ge_0_01"] = summary["delta_macro_f1_vs_A"].abs().ge(0.01)
summary.to_csv(SUMMARY_PATH, index=False)
display(summary.round(4))

per_class_rows = []
for name, frame in results.items():
    precision, recall, f1, support = precision_recall_fscore_support(
        frame["clause_type"],
        frame["predicted_label"],
        labels=CHOICES,
        zero_division=0,
    )
    for label, p, r, score, n in zip(CHOICES, precision, recall, f1, support):
        per_class_rows.append({
            "condition": name,
            "clause_type": label,
            "precision": p,
            "recall": r,
            "f1": score,
            "support": int(n),
        })

per_class = pd.DataFrame(per_class_rows)
per_class.to_csv(PER_CLASS_PATH, index=False)
focus = ["Agreement Date", "Effective Date", "License Grant", "Non-Transferable License"]
display(
    per_class[per_class["clause_type"].isin(focus)]
    .pivot(index="clause_type", columns="condition", values=["precision", "recall", "f1"])
    .round(4)
)

def transition_row(left_name, right_name):
    left = results[left_name]["predicted_label"].to_numpy()
    right = results[right_name]["predicted_label"].to_numpy()
    truth = val_df["clause_type"].to_numpy()
    left_correct = left == truth
    right_correct = right == truth
    return {
        "left": left_name,
        "right": right_name,
        "both_correct": int((left_correct & right_correct).sum()),
        "both_wrong": int((~left_correct & ~right_correct).sum()),
        "left_correct_right_wrong": int((left_correct & ~right_correct).sum()),
        "left_wrong_right_correct": int((~left_correct & right_correct).sum()),
        "prediction_changed": int((left != right).sum()),
    }


transitions = pd.DataFrame([
    transition_row("A_existing_label_prompt_default_bos", "B_label_prompt_single_bos"),
    transition_row("B_label_prompt_single_bos", "C_training_prompt_single_bos"),
    transition_row("A_existing_label_prompt_default_bos", "C_training_prompt_single_bos"),
])
transitions.to_csv(TRANSITIONS_PATH, index=False)
display(transitions)

print("Saved:")
print(SUMMARY_PATH)
print(PER_CLASS_PATH)
print(TRANSITIONS_PATH)


## 5. Exact-text conflict diagnostic on validation


In [ ]:
label_counts_by_text = val_df.groupby("clause_text")["clause_type"].nunique()
conflicting_texts = set(label_counts_by_text[label_counts_by_text > 1].index)
conflict_flag = val_df["clause_text"].isin(conflicting_texts).to_numpy()

conflict_rows = []
for condition_name, frame in results.items():
    for subset_name, mask in [
        ("all", np.ones(len(val_df), dtype=bool)),
        ("exact_text_conflict", conflict_flag),
        ("no_exact_text_conflict", ~conflict_flag),
    ]:
        subset = frame.loc[mask]
        present_labels = sorted(subset["clause_type"].unique())
        conflict_rows.append({
            "condition": condition_name,
            "subset": subset_name,
            "rows": int(len(subset)),
            "labels_present": int(len(present_labels)),
            "accuracy": accuracy_score(subset["clause_type"], subset["predicted_label"]),
            "macro_f1_present_labels": f1_score(
                subset["clause_type"],
                subset["predicted_label"],
                labels=present_labels,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                subset["clause_type"],
                subset["predicted_label"],
                labels=present_labels,
                average="weighted",
                zero_division=0,
            ),
        })

conflict_analysis = pd.DataFrame(conflict_rows)
conflict_analysis.to_csv(CONFLICT_PATH, index=False)
display(conflict_analysis.round(4))

ceiling = (
    val_df.groupby(["clause_text", "clause_type"])
    .size()
    .groupby(level=0)
    .max()
    .sum()
    / len(val_df)
)
print(f"Conflicting validation texts: {len(conflicting_texts)}")
print(f"Rows carrying a conflicted exact text: {int(conflict_flag.sum())}/{len(val_df)}")
print(f"Exact-text deterministic accuracy ceiling: {ceiling:.4f}")
print("Saved:", CONFLICT_PATH)


## 6. Decision record and hard stop


In [ ]:
best_row = summary.sort_values(
    ["macro_f1", "accuracy"], ascending=[False, False]
).iloc[0]

metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "validation-only BOS and prompt-match diagnostic for completed PEFT v4 adapter",
    "adapter": ADAPTER_DIR,
    "model_id": MODEL_ID,
    "conditions": {
        "A": "existing label-list prompt with original/default tokenization",
        "B": "same label-list prompt with add_special_tokens=False",
        "C": "exact v4 training prompt with add_special_tokens=False",
    },
    "selection_primary": "validation macro F1",
    "materiality_rule": "absolute macro-F1 change >= 0.01 or a major class-level recovery/collapse",
    "best_validation_condition_by_macro_f1": best_row["condition"],
    "best_validation_macro_f1": float(best_row["macro_f1"]),
    "test_loaded": False,
    "test_evaluated": False,
    "retraining_performed": False,
}
with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print(json.dumps(metadata, indent=2))
print("\nHARD STOP: do not add test evaluation or retraining to this notebook.")
print("Send back the summary, focused per-class table and transitions table for review.")
